# ROGII Wellbore TVT — **Stacked-UNet ENSEMBLE** Inference · per-member (K, handoff_T)

Multi-model fork of `stacked-unet-inference.ipynb`. Pure-model arm (no spatial blend). The
per-member machinery (channels, PhaseJitter TTA, expected-row decode, batched ckpt loading,
submission build) is unchanged — the change is an outer loop over a `MODEL_LIST` registry with a
weighted blend of the member outputs:

1. **`MODEL_LIST` registry** — each member declares its own artifact dir, global blend `weight`,
   SMP encoder string (`smp_encoder`, default = read from its own manifest), refeed depth
   `eval_k`, and belief temperatures `handoff_t` / `refeed_t`. Temperatures are cross-checked
   against each member's OWN manifest — hard fail on mismatch, loud warning if the manifest
   predates the fields.
2. **Weight guard** — member weights must sum to **exactly** 1.0 (float-exact `!=` check, no
   tolerance); a 50/50 blend of two members is `weight=0.5` each. Common splits (0.5/0.5,
   0.25/0.75, thirds) sum exactly; if an exotic split trips the guard, define the last member's
   weight as `1.0 - sum(others)` so the total is exact by construction.
3. **Blend semantics** — convex combination at the PREDICTION level (per-MD-row TVT, after each
   member's own ckpt-mean + TTA). A member's weight applies to its ensemble MEAN, so members with
   different ckpt counts contribute by declared weight, not ckpt count.
4. **Per-member temperatures live on the model instance** (`model.handoff_t` / `model.refeed_t`),
   not module globals — two members at different T coexist in one process. Everything downstream
   of the per-member mean is linear, so batched ckpt loading remains identical to all-at-once.

Example: one member at `eval_k=1` (single stage1→stage2 pass) + one at `eval_k=3` (K=3 refeed),
both at `handoff_t=1.5`, blended 0.5/0.5.

- **Per-member image geometry (2026-08-03):** each member renders at its OWN `H`/`W`/`BIN_SIZE` (from its manifest `image_constants`; spec override cross-checked, hard-fail on disagreement). Decoded per-MD-row predictions are geometry-free, so mixed-geometry members blend directly.


In [ ]:
!pip install -q /kaggle/input/datasets/tuckerarrants/smp-offline/smp_offline_wheels/smp_offline_wheels/segmentation_models_pytorch-0.5.0-py3-none-any.whl

In [ ]:
!pip install -q /kaggle/input/datasets/tuckerarrants/smp-offline/smp_offline_wheels/smp_offline_wheels/timm-1.0.27-py3-none-any.whl

In [ ]:
import os, sys, json, glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import segmentation_models_pytorch as smp

# DEFAULT image geometry — members may override PER MEMBER (see MODEL_LIST fields below).
# Each member renders its OWN images at its manifest geometry; only the decoded per-MD-row
# predictions (geometry-free) meet at the blend. The bare names remain for any default-geometry
# code path; IN_CH stays kernel-locked (all artifacts are 5-channel).
DEFAULT_H = 256
DEFAULT_W = 512
DEFAULT_BIN_SIZE = 1.0
DEFAULT_K_MAX = 12
H, W, BIN_SIZE, K_MAX = DEFAULT_H, DEFAULT_W, DEFAULT_BIN_SIZE, DEFAULT_K_MAX
IN_CH = 5

STACK_DETACH = True  # matches training; numerically INERT at inference (no_grad makes .detach() a no-op)
CKPT_DEVICE  = None  # None -> DEVICE (ckpts resident on GPU). Set 'cpu' to stream each model GPU<-CPU per forward.

# =============================================================================================
# === MODEL_LIST — the ensemble registry ======================================================
# =============================================================================================
# One entry per member. Fields:
#   ckpt_dir    (str)            REQUIRED. Kaggle dataset dir holding the member's manifest
#                                (*_kaggle_inference_manifest.json) + ckpt_f{f}_s{s}*.pt files.
#                                Searched one level, then recursively (Kaggle nests uploads).
#   weight      (float)          REQUIRED. Global blend weight for this member's OUTPUT (its
#                                ckpt-mean, TTA-averaged prediction). All weights must sum to
#                                EXACTLY 1.0 — hard error otherwise (see guard below).
#   smp_encoder (str or None)    SMP encoder name string controlling the UNet backbone (both
#                                stages). None -> read from the member's manifest (recommended).
#                                If set AND the manifest disagrees -> hard fail (the manifest is
#                                the artifact's source of truth; a disagreeing override builds an
#                                arch the checkpoints cannot load).
#   eval_k      (int)            Refeed depth for THIS member. 1 = single stage1->stage2 pass;
#                                3 = K=3 self-refeed (the deployment depth of the parent kernel).
#   handoff_t   (float)          Stage1->stage2 handoff temperature: softmax(logits1 / T). Live
#                                at EVERY k, k=1 included. MUST match the member's training run
#                                (cross-checked against its manifest, hard fail on mismatch).
#   refeed_t    (float, opt)     Stage-2 self-refeed temperature on rounds 2..k. Default 1.0.
#                                Inert at eval_k=1. Cross-checked against the manifest too.
#   folds       (list or None)   Which CV folds' ckpts to use for this member. None -> all folds
#                                in the member's manifest.
#   H, W        (int or None)    Per-member image geometry. None (recommended) -> read from the
#   bin_size    (float or None)  member's manifest image_constants. If set AND the manifest
#                                disagrees -> hard fail (same doctrine as smp_encoder: the
#                                manifest is the artifact's source of truth; a wrong geometry
#                                renders images the checkpoints were never trained on and
#                                decodes at the wrong vertical scale, silently).
#                                House formula for reference: W = 2*H, bin_size = 256/H
#                                (the vertical extent is fixed at 256 ft; e.g. H=224 ->
#                                W=448, bin_size = 256/224 ~= 1.142857).
#
# The two example members below: identical recipe (encoder from manifest, handoff_t=1.5) except
# the refeed depth — one pure stage1->stage2 (k=1), one K=3 refeed — blended 50/50.
MODEL_LIST = {
    'l2_hT15_k1': dict(
        ckpt_dir   = '/kaggle/input/models/tuckerarrants/rogii-l2/pytorch/default/1/5fold_l2_stack2_ema_25000pt_mdwarp42_bedwarp42_handoffT15_expa_pt',
        weight     = 0.25,
        smp_encoder= None,      
        eval_k     = 1,
        handoff_t  = 1.5,
        refeed_t   = 1.0,
        folds      = [0, 1, 2, 3, 4],
        H          = None,      
        W          = None,
        bin_size   = None,
    ),
    'xl_hT15_k3': dict(
        ckpt_dir   = '/kaggle/input/datasets/tuckerarrants/rogii-v2s/5fold_effnetv2_xl_stack2_ema_25000pt_mdwarp36_bedwarp36_unroll50_deepsup_handoffT15_expa_pt/5fold_effnetv2_xl_stack2_ema_25000pt_mdwarp36_bedwarp36_unroll50_deepsup_handoffT15_expa_pt',
        weight     = 0.25,
        smp_encoder= None,      
        eval_k     = 3,
        handoff_t  = 1.5,
        refeed_t   = 1.0,
        folds      = [0, 1, 2, 3, 4],
        H          = None,      # -> manifest image_constants (recommended)
        W          = None,     
        bin_size   = None,
    ),
    'l_hT15_k3': dict(
        ckpt_dir   = '/kaggle/input/datasets/tuckerarrants/rogii-v2s/5fold_effnetv2_l_stack2_ema_25000pt_mdwarp36_bedwarp36_unroll50_deepsup_handoffT15_expa_pt/5fold_effnetv2_l_stack2_ema_25000pt_mdwarp36_bedwarp36_unroll50_deepsup_handoffT15_expa_pt',
        weight     = 0.25,
        smp_encoder= None,
        eval_k     = 3,         
        handoff_t  = 1.5,
        refeed_t   = 1.0,
        folds      = [0, 1, 2, 3, 4],
        H          = None,     
        W          = None,      
        bin_size   = None,
    ),
    'l_hT15_k3_small': dict(
        ckpt_dir   = '/kaggle/input/datasets/tuckerarrants/rogii-v2s/5fold_effnetv2_l_stack2_ema_25000pt_mdwarp36_bedwarp36_unroll50_deepsup_handoffT15_expa_pt_H224_W448_bin1.1428571428571428/5fold_effnetv2_l_stack2_ema_25000pt_mdwarp36_bedwarp36_unroll50_deepsup_handoffT15_expa_pt_H224_W448_bin1.1428571428571428',
        weight     = 0.25,
        smp_encoder= None,
        eval_k     = 3,         
        handoff_t  = 1.5,
        refeed_t   = 1.0,
        folds      = [0, 1, 2, 3, 4],
        H          = None,      
        W          = None,     
        bin_size   = None,
    ),
}

# === WEIGHT GUARD — blend weights must sum to EXACTLY 1.0 ====================================
# Float-exact by request: `sum(...) != 1.0` raises, no epsilon. Common splits (0.5/0.5,
# 0.25/0.75, thirds) sum exactly in double precision; if an exotic split trips the guard, define
# the last weight as 1.0 - sum(others) so the total is exact by construction.
def _assert_weights_sum_to_one(model_list):
    if len(model_list) == 0:
        raise ValueError('MODEL_LIST is empty — the ensemble needs at least one member.')
    ws = {name: float(spec['weight']) for name, spec in model_list.items()}
    bad = {n: w for n, w in ws.items() if not np.isfinite(w) or not (0.0 < w <= 1.0)}
    if bad:
        raise ValueError(f'MODEL_LIST weights must each be finite and in (0, 1]: bad entries {bad}')
    total = float(sum(ws.values()))
    if total != 1.0:
        raise ValueError(
            f'MODEL_LIST weights must sum to EXACTLY 1.0 — got {total!r} from {ws}.\n'
            f'A 50/50 blend of two members is weight=0.5 each. If an intended-exact split '
            f'trips this, set the final member to 1.0 - sum(of the others).')
    return ws

BLEND_WEIGHTS = _assert_weights_sum_to_one(MODEL_LIST)
print('MODEL_LIST members and blend weights (sum == 1.0 verified, float-exact):')
for _n, _s in MODEL_LIST.items():
    print(f"  [{_n}] w={_s['weight']}  eval_k={_s['eval_k']}  handoff_t={_s['handoff_t']}  "
          f"refeed_t={_s.get('refeed_t', 1.0)}  folds={_s.get('folds')}  "
          f"phase_tta={_s.get('phase_tta', True)}")

# Basic per-member sanity that doesn't need the manifest yet
for _n, _s in MODEL_LIST.items():
    assert 'ckpt_dir' in _s and isinstance(_s['ckpt_dir'], str), f'{_n}: ckpt_dir (str) is required'
    assert int(_s['eval_k']) >= 1, f'{_n}: eval_k must be >= 1'
    assert float(_s['handoff_t']) > 0 and float(_s.get('refeed_t', 1.0)) > 0, \
        f'{_n}: temperatures must be positive'
    _f = _s.get('folds')
    if _f is not None:
        assert len(_f) > 0 and len(set(_f)) == len(_f), \
            f'{_n}: folds must be None or a non-empty unique list'
    for _gk in ('H', 'W'):
        _gv = _s.get(_gk)
        assert _gv is None or (int(_gv) > 0 and int(_gv) % 32 == 0), \
            f'{_n}: {_gk} must be None or a positive multiple of 32 (UNet stride), got {_gv}'
    _bs = _s.get('bin_size')
    assert _bs is None or float(_bs) > 0, f'{_n}: bin_size must be None or > 0'

# Locate competition data — try both common Kaggle path conventions
CANDIDATE_COMP_ROOTS = [
    '/kaggle/input/rogii-wellbore-geology-prediction',
    '/kaggle/input/competitions/rogii-wellbore-geology-prediction',
]
COMP_ROOT = None
for cand in CANDIDATE_COMP_ROOTS:
    if os.path.exists(cand):
        COMP_ROOT = cand
        break
if COMP_ROOT is None:
    raise FileNotFoundError(f'Could not find competition data. Tried: {CANDIDATE_COMP_ROOTS}')
print(f'Competition data at: {COMP_ROOT}')
TEST_DIR  = os.path.join(COMP_ROOT, 'test')
SAMPLE_SUB = os.path.join(COMP_ROOT, 'sample_submission.csv')

# === Device discovery + ensemble sharding (unchanged from the single-model kernel) ===========
N_GPU  = torch.cuda.device_count() if torch.cuda.is_available() else 0
GPUS   = [torch.device(f'cuda:{i}') for i in range(N_GPU)] if N_GPU else [torch.device('cpu')]
DEVICE = GPUS[0]                                  # inputs / intermediates default here
CKPT_DEVICE = DEVICE if CKPT_DEVICE is None else torch.device(CKPT_DEVICE)
SHARD_MODELS = (N_GPU >= 2) and (str(CKPT_DEVICE) == str(DEVICE))
print(f'Visible GPUs: {N_GPU} -> {GPUS}')
print(f'Compute device: {DEVICE} | shard ensemble across GPUs: {SHARD_MODELS}')
print(f'Checkpoint residency mode: {CKPT_DEVICE}')


In [ ]:
# Resolve + validate every MODEL_LIST member against its OWN manifest.
# The manifest is the source of truth per artifact: constants, encoder, stack depth, and (on
# newer runs) the training belief temperatures. Nothing loads weights here — paths only.
import re

def _find_manifest(ckpt_dir, name):
    """Locate the member's *_kaggle_inference_manifest.json — one level first, then recursive
    (Kaggle doubles-up upload dirs). Exactly one match required; ambiguity fails loudly."""
    if not os.path.isdir(ckpt_dir):
        raise FileNotFoundError(f"[{name}] ckpt_dir does not exist: {ckpt_dir}")
    hits = sorted(glob.glob(os.path.join(ckpt_dir, '*_kaggle_inference_manifest.json')))
    if not hits:
        hits = sorted(glob.glob(os.path.join(ckpt_dir, '**', '*_kaggle_inference_manifest.json'),
                                recursive=True))
    if len(hits) == 0:
        raise FileNotFoundError(
            f"[{name}] No *_kaggle_inference_manifest.json under {ckpt_dir}. "
            f"Dir contains: {sorted(os.listdir(ckpt_dir))[:10]}")
    if len(hits) > 1:
        raise RuntimeError(
            f"[{name}] {len(hits)} manifests under {ckpt_dir} — ambiguous member artifact:\n  "
            + '\n  '.join(hits) + '\nPoint ckpt_dir at the specific artifact folder.')
    return hits[0]

def _entry_fold(entry):
    """Fold id of a manifest checkpoint entry. Prefer an explicit 'fold' field; fall back to
    parsing the ckpt_f{f}_s{s}.pt filename convention. Fails loudly if neither is available."""
    if 'fold' in entry:
        return int(entry['fold'])
    m = re.search(r'_f(\d+)_s', entry['filename'])
    if m is None:
        raise ValueError(f"Cannot determine fold for manifest entry {entry['filename']!r}: "
                         f"no 'fold' field and filename does not match *_f{{fold}}_s{{seed}}*.pt")
    return int(m.group(1))

RESOLVED = {}   # name -> fully-validated member record (the walk loop consumes ONLY this)
for name, spec in MODEL_LIST.items():
    manifest_path = _find_manifest(spec['ckpt_dir'], name)
    member_dir = os.path.dirname(manifest_path)
    with open(manifest_path, 'r') as fh:
        manifest = json.load(fh)
    rec = manifest['recipe']
    print(f"\n=== [{name}] {manifest.get('experiment_id', '?')} ===")
    print(f"  artifact dir: {member_dir}")
    print(f"  n_checkpoints (manifest): {manifest['n_checkpoints']}")
    if 'ens_rmse' in manifest.get('full_cv_metric', {}):
        print(f"  full-CV ens TVT-RMSE (training estimate): {manifest['full_cv_metric']['ens_rmse']:.3f}")

    # --- PER-MEMBER image geometry: the member's manifest is the source of truth -------------
    # Spec overrides (H/W/bin_size) are cross-checked exactly like smp_encoder: a disagreeing
    # override renders images the checkpoints never saw and decodes at the wrong vertical
    # scale, silently — hard fail. IN_CH stays kernel-locked.
    _ic = manifest.get('image_constants')
    assert _ic is not None and all(k in _ic for k in ('H', 'W', 'BIN_SIZE', 'K_MAX')), \
        f'[{name}] manifest has no complete image_constants block — cannot resolve geometry'
    geom = dict(H=int(_ic['H']), W=int(_ic['W']),
                BIN_SIZE=float(_ic['BIN_SIZE']), K_MAX=int(_ic['K_MAX']))
    for _sk, _mk, _cast in (('H', 'H', int), ('W', 'W', int), ('bin_size', 'BIN_SIZE', float)):
        _ov = spec.get(_sk)
        if _ov is not None and _cast(_ov) != geom[_mk]:
            raise AssertionError(
                f"[{name}] {_sk}={_ov!r} but the manifest image_constants say "
                f"{_mk}={geom[_mk]!r}. The manifest is the artifact's source of truth — an "
                f"overriding geometry renders inputs these checkpoints were never trained on. "
                f"Fix the spec (or set {_sk}=None to inherit).")
    assert geom['H'] % 32 == 0 and geom['W'] % 32 == 0, \
        f"[{name}] geometry H={geom['H']}, W={geom['W']} not divisible by 32 (UNet stride)"
    _vext = geom['H'] * geom['BIN_SIZE']
    if abs(_vext - 256.0) > 1e-6:
        print(f"  NOTE: [{name}] vertical extent H*BIN_SIZE = {_vext:.4f} ft != the house 256 ft "
              f"— unusual; double-check this artifact's training render.")
    if geom['W'] != 2 * geom['H']:
        print(f"  NOTE: [{name}] W={geom['W']} != 2*H={2*geom['H']} — aspect differs from the "
              f"house formula; double-check.")
    print(f"  geometry: H={geom['H']} W={geom['W']} BIN_SIZE={geom['BIN_SIZE']:.6f} "
          f"K_MAX={geom['K_MAX']}  (vertical extent {_vext:.1f} ft)"
          + ('' if geom['H'] == DEFAULT_H else '  [non-default]'))
    assert rec['in_ch'] == IN_CH, f"[{name}] manifest in_ch={rec['in_ch']} != kernel IN_CH={IN_CH}"

    # --- Encoder (the SMP model-name string): manifest is authoritative ----------------------
    encoder_name = rec['encoder_name']
    if spec.get('smp_encoder') is not None and spec['smp_encoder'] != encoder_name:
        raise AssertionError(
            f"[{name}] smp_encoder={spec['smp_encoder']!r} but the manifest says "
            f"{encoder_name!r}. The manifest is the artifact's source of truth — an overriding "
            f"encoder builds an architecture these checkpoints cannot load. Fix the spec (or "
            f"set smp_encoder=None to inherit).")

    # --- Stack depth ------------------------------------------------------------------------
    stack_n = int(rec.get('stack_n', 1))
    eval_k  = int(spec['eval_k'])
    assert stack_n >= 2 or eval_k == 1, (
        f'[{name}] eval_k={eval_k} needs a stacked artifact but manifest stack_n={stack_n}')
    if stack_n >= 2:
        print(f"  stacked artifact: stack_n={stack_n}, handoff={rec.get('stack_handoff')}, "
              f"stage2_in_ch={IN_CH + 1}")

    # --- Belief-temperature cross-check (per member, against ITS manifest) -------------------
    # Lookup order per key:
    #   1) structured field ('handoff_T' / 'refeed_T' in recipe) — exact, wins outright;
    #   2) free-text mining of the whole serialized manifest, which catches the channel_note
    #      convention ("... + handoffT=1.5. All augs are finetune-train-only ...").
    # Found (either way) -> HARD FAIL on mismatch. Ambiguous text (>=2 DISTINCT values) ->
    # treated as unverifiable with a loud warning. Truly absent -> loud warning: the kernel is
    # then trusting the hand-set spec values with nothing to verify them against. A mismatch
    # loads cleanly and silently decodes at the wrong temperature — CV/LB divergence, no error.
    _manifest_text = json.dumps(manifest)
    _T_PATTERNS = {
        'handoff_T': re.compile(r'\bhandoff[_\- ]?T\s*[=:]\s*([0-9]+(?:\.[0-9]+)?)', re.I),
        'refeed_T':  re.compile(r'\brefeed[_\- ]?T\s*[=:]\s*([0-9]+(?:\.[0-9]+)?)',  re.I),
    }   # NB: number group excludes a trailing sentence period ("handoffT=1.5." -> 1.5)

    def _recorded_temp(_key):
        """-> (value, source) | (None, reason). Structured field first, then note mining."""
        if _key in rec:
            return float(rec[_key]), 'recipe field'
        _vals = sorted({float(v) for v in _T_PATTERNS[_key].findall(_manifest_text)})
        if len(_vals) == 1:
            return _vals[0], 'free-text note'
        if len(_vals) > 1:
            return None, f'AMBIGUOUS in free text: {_vals}'
        return None, 'absent'

    handoff_t = float(spec['handoff_t'])
    refeed_t  = float(spec.get('refeed_t', 1.0))
    _mism = []
    for _key, _local in (('handoff_T', handoff_t), ('refeed_T', refeed_t)):
        _rec_val, _src = _recorded_temp(_key)
        if _rec_val is not None:
            if float(_rec_val) != float(_local):
                _mism.append(f'{_key}: manifest={_rec_val} ({_src}) spec={_local}')
            else:
                print(f"  {_key} verified against manifest ({_src}): {_rec_val}")
        else:
            _train_name = _key.upper().replace('_T', '_TRAIN_T')
            print(f"  WARNING: [{name}] {_key} not verifiable from manifest ({_src}). Confirm "
                  f"by hand against this member's training notebook that {_train_name} == {_local}.")
    if _mism:
        raise AssertionError(
            f'[{name}] Belief-temperature MISMATCH between spec and training manifest:\n  '
            + '\n  '.join(_mism)
            + '\nThe checkpoints were trained AND validated at the manifest values. Running at '
              'any other temperature decodes a distribution the model never saw, silently. Fix '
              'handoff_t / refeed_t in MODEL_LIST.')
    if 'checkpoint_k' in rec and int(rec['checkpoint_k']) != eval_k:
        print(f"  NOTE: [{name}] manifest checkpoint_k={rec['checkpoint_k']} but eval_k={eval_k}. "
              f"Deliberate depth sweeps are fine, but this member's stream then differs from "
              f"its selection metric.")
    print(f"  belief temps: handoff_t={handoff_t} (every k)  refeed_t={refeed_t} "
          f"({'live' if eval_k >= 2 else 'INERT at eval_k=1'})")

    # --- Fold filter ------------------------------------------------------------------------
    _all_folds = sorted({_entry_fold(e) for e in manifest['checkpoints']})
    folds = spec.get('folds')
    folds = list(_all_folds) if folds is None else sorted(folds)
    _unknown = sorted(set(folds) - set(_all_folds))
    assert not _unknown, \
        f'[{name}] folds={folds} requests folds not in the manifest (available: {_all_folds})'
    ckpt_entries = [e for e in manifest['checkpoints'] if _entry_fold(e) in set(folds)]
    assert len(ckpt_entries) > 0, f'[{name}] fold filter folds={folds} matched no checkpoints'
    print(f"  fold filter: {folds} of available {_all_folds} -> "
          f"{len(ckpt_entries)}/{len(manifest['checkpoints'])} checkpoints selected")

    # --- Resolve (and validate existence of) every SELECTED ckpt path WITHOUT loading --------
    ckpt_paths = []
    for entry in ckpt_entries:
        ckpt_path = os.path.join(member_dir, entry['filename'])
        if not os.path.exists(ckpt_path):
            # REUSE_PT saves ckpt_f{f}_s{s}_*.pt; the manifest omits the suffix.
            _hits = sorted(glob.glob(os.path.join(member_dir, entry['filename'][:-3] + '_*.pt')))
            if not _hits:
                raise FileNotFoundError(
                    f"[{name}] No checkpoint matching {entry['filename'][:-3]}_*.pt in "
                    f"{member_dir}. Dir contains: {sorted(os.listdir(member_dir))[:10]}")
            ckpt_path = _hits[0]
        ckpt_paths.append(ckpt_path)
    assert len(ckpt_paths) == len(ckpt_entries), \
        f'[{name}] resolved path count != selected entry count'
    ckpt_seeds = [int(entry['seed']) for entry in ckpt_entries]

    RESOLVED[name] = dict(
        name=name, weight=float(spec['weight']), ckpt_dir=member_dir, manifest=manifest,
        encoder_name=encoder_name, stack_n=stack_n, eval_k=eval_k,
        handoff_t=handoff_t, refeed_t=refeed_t, folds=folds, geom=geom,
        ckpt_paths=ckpt_paths, ckpt_seeds=ckpt_seeds)
    print(f"  RESOLVED: {len(ckpt_paths)} ckpts | encoder={encoder_name} | weight={spec['weight']}")

# Defense in depth: re-verify the float-exact weight sum on the RESOLVED registry the walk uses.
_total = float(sum(e['weight'] for e in RESOLVED.values()))
if _total != 1.0:
    raise ValueError(f'RESOLVED member weights must sum to EXACTLY 1.0 — got {_total!r}')
print(f'\nAll {len(RESOLVED)} members resolved; blend weights sum to 1.0 (float-exact) — OK')


In [ ]:
# === Model definition — inference twin of the training StackedUNet ============================
# ENSEMBLE-FORK CHANGE: HANDOFF_T / REFEED_T are no longer module globals — each model instance
# carries its OWN handoff_t / refeed_t (set at build time from the member's RESOLVED record), so
# two members trained at different temperatures coexist in one process. forward() and
# _iter_logits read the instance attributes; the math is otherwise line-for-line the parent
# kernel's (which is itself the twin of the training _eval_iter_logits).
#
# RETIRED AND REMOVED (2026-07-27), not merely switched off (inherited from the parent kernel):
#   * POSTERIOR MD-DILATION and PIXELSHUFFLE / SUB-PIXEL DECODER — a checkpoint carrying
#     post_md_dilation.* or subpixel_upsamplers.* keys fails loudly at load_state_dict.

DECODER_INTERP   = "nearest"                   # native SMP decoder upsampling; matches training
DECODER_CHANNELS = (512, 256, 128, 64, 32)     # matches training DECODER_CHANNELS


class UNetGeneric(nn.Module):
    """Thin wrapper around smp.Unet (stack_n==1 artifacts). State-dict keys are prefixed smp.*.
    Supports eval_k=1 only; temperatures are inert but stored for a uniform interface."""
    def __init__(self, encoder_name='resnet34', in_ch=5, handoff_t=1.0, refeed_t=1.0):
        super().__init__()
        self.handoff_t = float(handoff_t)
        self.refeed_t  = float(refeed_t)
        self.smp = smp.Unet(encoder_name=encoder_name, encoder_weights=None,
                            in_channels=in_ch, classes=1,
                            decoder_interpolation=DECODER_INTERP)
    def forward(self, x):
        return self.smp(x).squeeze(1)


class StackedUNet(nn.Module):
    """Two-stage ("stacked") UNet — inference twin of the training StackedUNet.

    Stage 1 maps the IN_CH inputs to a per-column row-logit map -> softmax over H = a
    row-posterior. Stage 2 re-ingests the ORIGINAL IN_CH inputs concatenated with the stage-1
    posterior plane (IN_CH+1 channels) and emits the refined row-logit map.

    The stage-1 -> stage-2 handoff is softened by self.handoff_t, exactly as the training
    forward() does with HANDOFF_TRAIN_T. That temperature is live on EVERY forward in training —
    it shaped the weights AND the CV number — so it must be applied here at every k, k=1
    included. It is a PER-MEMBER value in this kernel.

    forward(x) returns the STAGE-2 logits (B, H, W) — identical contract to UNetGeneric, so the
    channel builder, softmax-expected-row decode, PhaseJitter TTA and ensembling are unchanged.

    State-dict keys are stage1.* / stage2.* and MUST match the training StackedUNet. detach is
    value-inert under no_grad."""
    def __init__(self, encoder_name='efficientnet-b7', in_ch=5, detach_handoff=True,
                 handoff_t=1.0, refeed_t=1.0):
        super().__init__()
        self.detach_handoff = detach_handoff
        self.handoff_t = float(handoff_t)
        self.refeed_t  = float(refeed_t)
        self.stage1 = smp.Unet(encoder_name=encoder_name, encoder_weights=None,
                               in_channels=in_ch, classes=1,
                               decoder_channels=DECODER_CHANNELS,
                               decoder_interpolation=DECODER_INTERP)
        self.stage2 = smp.Unet(encoder_name=encoder_name, encoder_weights=None,
                               in_channels=in_ch + 1, classes=1,
                               decoder_channels=DECODER_CHANNELS,
                               decoder_interpolation=DECODER_INTERP)

    def _stage2_from_post(self, x, post):
        """One stage-2 pass given input x and a row-posterior `post` (B,H,W). Mirrors the
        training _stage2_from_post so an iterated round is identical to round 1 except for the
        belief fed."""
        return self.stage2(torch.cat([x, post.unsqueeze(1)], dim=1)).squeeze(1)

    def forward(self, x, return_aux=False):
        logits1 = self.stage1(x).squeeze(1)                  # (B, H, W)
        handoff = logits1.detach() if self.detach_handoff else logits1
        post1   = F.softmax(handoff / self.handoff_t, dim=1) # row-posterior; /T == training forward()
        logits2 = self._stage2_from_post(x, post1)           # (B, H, W) — refined, headline output
        if return_aux:
            return logits2, logits1
        return logits2


def build_model(entry):
    """Single source of truth for inference model construction, driven by a RESOLVED member
    record: stack_n>=2 -> StackedUNet at the member's temperatures; else UNetGeneric."""
    if entry['stack_n'] >= 2:
        return StackedUNet(encoder_name=entry['encoder_name'], in_ch=IN_CH,
                           detach_handoff=STACK_DETACH,
                           handoff_t=entry['handoff_t'], refeed_t=entry['refeed_t'])
    return UNetGeneric(encoder_name=entry['encoder_name'], in_ch=IN_CH,
                       handoff_t=entry['handoff_t'], refeed_t=entry['refeed_t'])


def _iter_logits(model, x, k):
    """k-round refeed iteration — line-for-line twin of the training _eval_iter_logits, at the
    MODEL'S OWN temperatures:

        logits1 = stage1(x)
        post    = softmax(logits1 / model.handoff_t)     <- handoff temp, every k
        out[1]  = stage2(cat(x, post))
        for j in 2..k:
            post   = softmax(out[j-1] / model.refeed_t)  <- refeed temp, rounds 2..k
            out[j] = stage2(cat(x, post))

    Returns {j: stage-2 logits} for j=1..k; j==1 is exactly forward()'s computation."""
    if not isinstance(model, StackedUNet):
        assert k == 1, 'single-UNet path supports k=1 only'
        return {1: model(x)}
    logits1 = model.stage1(x).squeeze(1)
    post = F.softmax(logits1 / model.handoff_t, dim=1)       # handoff softened, as in training
    out = {}
    logits2 = model._stage2_from_post(x, post)
    out[1] = logits2
    for j in range(2, k + 1):
        post = F.softmax(logits2 / model.refeed_t, dim=1)    # re-feed stage-2's OWN belief, softened
        logits2 = model._stage2_from_post(x, post)
        out[j] = logits2
    return out


print(f'Model classes ready | decoder_interp={DECODER_INTERP} | decoder_channels={DECODER_CHANNELS}')
print('  posterior MD-dilation: REMOVED   |   pixelshuffle/sub-pixel decoder: REMOVED')
print('  temperatures are PER-INSTANCE (member-owned):')
for _n, _e in RESOLVED.items():
    _arch = 'StackedUNet' if _e['stack_n'] >= 2 else 'UNetGeneric'
    print(f"    [{_n}] {_arch} encoder={_e['encoder_name']} handoff_t={_e['handoff_t']} "
          f"refeed_t={_e['refeed_t']} eval_k={_e['eval_k']}")


In [ ]:
# === dZ-per-column channel helpers (FT39e) — VERBATIM copy from training cell 12 ===
def _dz_per_col(hw_z, hw_md, bin_edges, hw_mask, W=None):
    """Per-column average of dZ/dMD. Returns float32 (W,) with NaN where no data."""
    if W is None: W = DEFAULT_W
    n = len(hw_z)
    if n < 2:
        return np.full(W, 0.0, dtype=np.float32)
    # uniform 1.0 ft MD step per the EDA report, but compute defensively
    dz_per_row = np.gradient(np.asarray(hw_z, dtype=np.float64),
                              np.asarray(hw_md, dtype=np.float64))
    dz = np.full(W, np.nan, dtype=np.float32)
    for c in range(W):
        lo, hi = bin_edges[c], bin_edges[c+1]
        if hi <= lo:
            continue
        dz[c] = float(np.mean(dz_per_row[lo:hi]))
    return dz

def _dz_channel(dz_per_col, hw_mask, H=None, W=None):
    """Build the (H, W) dZ channel — per-well z-score + clip ±3 + broadcast across H.
    Neutralize columns without HW data so they don't bias the well z-score / clip."""
    if H is None: H = DEFAULT_H
    if W is None: W = DEFAULT_W
    bad = (~hw_mask) | (~np.isfinite(dz_per_col))
    if (~bad).any():
        fill = float(np.nanmean(dz_per_col[~bad]))
    else:
        fill = 0.0
    dz = np.where(bad, fill, dz_per_col).astype(np.float32)
    m, s = dz.mean(), dz.std()
    dz_norm = (dz - m) / s if s > 1e-6 else np.zeros_like(dz)
    dz_norm = np.clip(dz_norm, -3.0, 3.0)
    return np.broadcast_to(dz_norm[None, :], (H, W)).copy().astype(np.float32)


def build_image_from_arrays(hw_md, hw_gr, hw_z, hw_tvt, hw_tvti, tw_tvt, tw_gr,
                            H=None, W=None, BIN_SIZE=None, K_MAX=None):
    '''Identical to FT39e cell 12 build_image_from_arrays (5-channel incl. dZ).

    For test wells, hw_tvt is all NaN (target not provided).
    The function uses hw_tvt only to compute truth_row, which is not needed at inference.

    GEOMETRY: H/W/BIN_SIZE/K_MAX are now PARAMETERS (None -> the kernel defaults). The
    parameter names shadow the module globals, so the body below is line-for-line the
    verbatim training build — only the dimensions it reads are per-call.
    '''
    if H is None: H = DEFAULT_H
    if W is None: W = DEFAULT_W
    if BIN_SIZE is None: BIN_SIZE = DEFAULT_BIN_SIZE
    if K_MAX is None: K_MAX = DEFAULT_K_MAX
    n_hw = len(hw_md)
    known_tvt = pd.Series(hw_tvti).dropna().values
    if len(known_tvt) == 0:
        raise ValueError('No pre-PS rows')
    depth_center = float(known_tvt[-1])
    tvt_lo = depth_center - (H/2) * BIN_SIZE

    def tvt_to_row(t, lo=tvt_lo):
        if np.isnan(t): return -1
        r = int((t - lo) / BIN_SIZE)
        return r if 0 <= r < H else -1

    bin_edges = np.linspace(0, n_hw, W + 1).astype(int)
    hw_gr_per_col_raw = np.full(W, np.nan, dtype=np.float32)
    hw_mask = np.zeros(W, dtype=bool)
    truth_row = np.full(W, -1, dtype=np.int32)
    hint_row = np.full(W, -1, dtype=np.int32)
    post_ps_mask = np.zeros(W, dtype=bool)

    for c in range(W):
        lo, hi = bin_edges[c], bin_edges[c+1]
        if hi <= lo: continue
        gr_slc = hw_gr[lo:hi]
        if np.any(~np.isnan(gr_slc)):
            hw_gr_per_col_raw[c] = np.nanmean(gr_slc)
            hw_mask[c] = True
        tvt_slc = hw_tvt[lo:hi]
        if np.any(~np.isnan(tvt_slc)):
            truth_row[c] = tvt_to_row(np.nanmedian(tvt_slc))
        tvti_slc = hw_tvti[lo:hi]
        is_known = ~np.isnan(tvti_slc)
        if is_known.any():
            hint_row[c] = tvt_to_row(np.nanmedian(tvti_slc[is_known]))
        else:
            post_ps_mask[c] = True

    if np.all(np.isnan(hw_gr_per_col_raw)):
        hw_gr_per_col = np.zeros(W, dtype=np.float32)
    else:
        col_mean = np.nanmean(hw_gr_per_col_raw)
        hw_gr_per_col = hw_gr_per_col_raw.copy()
        hw_gr_per_col[np.isnan(hw_gr_per_col)] = col_mean

    ext_row_tvts = (tvt_lo - K_MAX * BIN_SIZE) + (np.arange(H + 2*K_MAX) + 0.5) * BIN_SIZE
    sort_idx = np.argsort(tw_tvt)
    tw_tvt_s = tw_tvt[sort_idx]; tw_gr_s = tw_gr[sort_idx]
    ok = ~np.isnan(tw_gr_s)
    if ok.sum() < 2:
        tw_gr_per_row_ext = np.zeros(H + 2*K_MAX, dtype=np.float32)
    else:
        tw_gr_per_row_ext = np.interp(ext_row_tvts, tw_tvt_s[ok], tw_gr_s[ok],
                                       left=tw_gr_s[ok][0], right=tw_gr_s[ok][-1]).astype(np.float32)
    tw_gr_per_row = tw_gr_per_row_ext[K_MAX : K_MAX + H]

    ch_tw_raw = np.broadcast_to(tw_gr_per_row[:, None], (H, W)).copy()
    ch_hw_raw = np.broadcast_to(hw_gr_per_col[None, :], (H, W)).copy()
    ch_hint = np.zeros((H, W), dtype=np.float32)
    for c in range(W):
        if hint_row[c] >= 0 and not post_ps_mask[c]:
            ch_hint[hint_row[c], c] = 1.0

    cost_vol = (ch_hw_raw - ch_tw_raw) ** 2
    if hw_mask.any():
        mean_cost = float(cost_vol[:, hw_mask].mean())
    else:
        mean_cost = 0.0
    for c in range(W):
        if not hw_mask[c]:
            cost_vol[:, c] = mean_cost

    ch_tw = ch_tw_raw.copy()
    m, s = ch_tw.mean(), ch_tw.std()
    if s > 1e-6: ch_tw = (ch_tw - m) / s

    ch_hw = ch_hw_raw.copy()
    m, s = ch_hw.mean(), ch_hw.std()
    if s > 1e-6: ch_hw = (ch_hw - m) / s

    cm, cs = cost_vol.mean(), cost_vol.std()
    if cs > 1e-6:
        cost_vol_norm = (cost_vol - cm) / cs
    else:
        cost_vol_norm = cost_vol - cm
    cost_vol_norm = np.clip(cost_vol_norm, -3.0, 3.0)

    # --- dZ-per-column channel (FT39e) — VERBATIM stack order from training cell 12 ---
    dz_per_col = _dz_per_col(hw_z, hw_md, bin_edges, hw_mask, W=W)
    ch_dz = _dz_channel(dz_per_col, hw_mask, H=H, W=W)
    image_in = np.stack([ch_tw, ch_hw, ch_hint, cost_vol_norm, ch_dz], axis=0).astype(np.float32)
    return dict(image_in=image_in, truth_row=truth_row, hint_row=hint_row,
                post_ps_mask=post_ps_mask, depth_center=depth_center,
                tvt_lo=tvt_lo, hw_mask=hw_mask, bin_edges=bin_edges, dz_per_col=dz_per_col)


In [ ]:
# Batched checkpoint loading, per member — identical OOM strategy to the parent kernel: resolve
# all paths up front (done in the RESOLVED cell), load MODELS_PER_BATCH ckpts at a time inside
# the prediction loop, accumulate per-well sums, FREE each batch before the next. Members run
# sequentially, so peak residency is one batch of ONE member. Every downstream step is LINEAR,
# so batched accumulation == all-at-once (up to float summation order).
import gc
_BN_COUNTER = 'num_batches_tracked'   # int BN counters; eval-irrelevant, absent from float-only EMA shadow

# How many checkpoints to hold in memory simultaneously (applies within each member's loop).
# MODELS_PER_BATCH is now a CAP; the effective per-member batch is AUTO-SIZED from each
# member's actual checkpoint file size and the per-GPU budget:
#   per_gpu = floor((VRAM_USABLE_GB - ACT_RESERVE_GB) / (max ckpt GiB * 1.10))
#   batch   = min(MODELS_PER_BATCH, per_gpu * n_gpus)      [>= 1 always]
# ACT_RESERVE covers phase-batched activations (PHASE_CHUNK images through the net).
# Examples on 2xT4: L2 @3.6 GiB -> 2/GPU -> batch 4 (two loads for 7 seeds);
# L @<1 GiB and XL @~1.6 GiB -> all 7 in one batch.
MODELS_PER_BATCH = 8
VRAM_USABLE_GB   = 13.5     # T4 ~15.8 total minus CUDA context/headroom
ACT_RESERVE_GB   = 3.5      # activation reserve for PHASE_CHUNK-image forwards

def member_batch_size(entry):
    try:
        max_gib = max(os.path.getsize(p) for p in entry['ckpt_paths']) / 2**30
    except Exception:
        return max(1, min(MODELS_PER_BATCH, len(GPUS)))
    per_gpu = int((VRAM_USABLE_GB - ACT_RESERVE_GB) // (max_gib * 1.10))
    n = max(1, min(MODELS_PER_BATCH, max(per_gpu, 1) * max(len(GPUS), 1)))
    entry['_ckpt_gib'] = max_gib
    return n

def _load_model_batch(entry, paths):
    """Load a list of the member's checkpoints into eval models, sharded across GPUs. Fails LOUD
    on arch mismatch (an unexpected/missing key means this member's inference architecture !=
    its artifact)."""
    batch = []
    for j, ckpt_path in enumerate(paths):
        dev = GPUS[j % len(GPUS)] if SHARD_MODELS else CKPT_DEVICE
        try:    # mmap avoids the full host-RAM copy and speeds cold loads of large ckpts
            state = torch.load(ckpt_path, map_location='cpu', mmap=True)
        except (TypeError, RuntimeError):
            state = torch.load(ckpt_path, map_location='cpu')   # older torch / non-zip format
        m = build_model(entry).to(dev)
        missing, unexpected = m.load_state_dict(state, strict=False)
        if len(unexpected) != 0:
            raise RuntimeError(
                f"[{entry['name']}] {os.path.basename(ckpt_path)}: {len(unexpected)} checkpoint "
                f"keys did not map into the model (e.g. {list(unexpected)[:4]}). Inference "
                f"architecture does not match the artifact -- check stack_n "
                f"({entry['stack_n']}), encoder ({entry['encoder_name']}), in_ch ({IN_CH}).")
        bad_missing = [k for k in missing if not k.endswith(_BN_COUNTER)]
        if len(bad_missing) != 0:
            raise RuntimeError(
                f"[{entry['name']}] {os.path.basename(ckpt_path)}: {len(bad_missing)} non-BN "
                f"params missing from the checkpoint (e.g. {bad_missing[:4]}). Artifact is "
                f"incomplete for this architecture.")
        m.eval()
        batch.append(m)
        del state
    return batch

def _free_model_batch(batch):
    """Drop a batch's models and reclaim GPU + host memory before the next batch loads."""
    for m in batch:
        del m
    batch.clear()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

for _n, _e in RESOLVED.items():
    _mb = member_batch_size(_e)
    _nb = (len(_e['ckpt_paths']) + _mb - 1) // _mb
    print(f"[{_n}] {len(_e['ckpt_paths'])} ckpts (folds {_e['folds']}) -> {_nb} batches of "
          f"<= {MODELS_PER_BATCH} (never all resident at once).")
print('Sharding: ' + ('round-robin across ' + str(len(GPUS)) + ' GPU(s)' if SHARD_MODELS
                      else str(CKPT_DEVICE))
      + f'; compute device: {DEVICE}.')


In [ ]:
# === PhaseJitter TTA — k1..k{eval_k} streams, PER MEMBER =======================================
# Phase machinery unchanged: crop toe rows -> re-bin -> predict -> broadcast -> average in TVT
# space per MD row. The k-iterates share the stage-1 forward, so k1..k{eval_k-1} come free on the
# way to k{eval_k}; lower iterates are kept as free diagnostics. Every stream is phase-averaged
# IDENTICALLY (shared coverage). k_compute is now an ARGUMENT (each member passes its own eval_k)
# and the temperatures ride on the model instances.
DO_PHASE_JITTER_TTA = False
PHASE_TTA_OFFSETS = [0.0, 0.125, 0.25, 0.375, 0.5, 0.625, 0.75, 0.875]  # full-bin sweep

_row_idx_cache = {}
def _row_idx_for(dev, H=None):
    if H is None: H = DEFAULT_H
    key = (str(dev), int(H))
    t = _row_idx_cache.get(key)
    if t is None:
        t = torch.arange(int(H), device=dev, dtype=torch.float32)
        _row_idx_cache[key] = t
    return t

# --- shared phase-image cache: members with the SAME geometry reuse each well's rendered
# phase images instead of rebuilding them (the render depends only on well arrays + geometry
# + offset — it is member-independent). fp32, budgeted; past the budget it falls back to
# rebuilding (still exact, just slower). The walk cell purges a geometry's entries once its
# last member finishes and prints built/reused stats per member.
_IMG_CACHE = {}
_IMG_CACHE_BYTES = 0
IMG_CACHE_GB = 8.0            # hard budget for cached rendered images
_IMG_STATS = {'built': 0, 'hits': 0}
PHASE_CHUNK = 8               # phase images forwarded per batch; set 4 if the largest member
                              # OOMs at its geometry (chunking is exact)

def _get_phase_rec(rec_arrays, cache_key, geom_key, off, _H, _W, _BS, _KM, n_hw, bin_w):
    global _IMG_CACHE_BYTES
    key = (cache_key, geom_key, float(off)) if cache_key is not None else None
    if key is not None and key in _IMG_CACHE:
        _IMG_STATS['hits'] += 1
        return _IMG_CACHE[key]
    drop = int(round(float(off) * bin_w))
    drop = min(drop, n_hw - _W - 1) if n_hw - _W - 1 > 0 else 0
    end = n_hw - drop
    a = {k: (v[:end] if isinstance(v, np.ndarray) and v.shape[0] == n_hw else v)
         for k, v in rec_arrays.items()}
    try:
        rec = build_image_from_arrays(**a, H=_H, W=_W, BIN_SIZE=_BS, K_MAX=_KM)
    except Exception:
        return None
    _IMG_STATS['built'] += 1
    if key is not None:
        sz = rec['image_in'].nbytes + rec['bin_edges'].nbytes + 64
        if _IMG_CACHE_BYTES + sz <= IMG_CACHE_GB * 2**30:
            _IMG_CACHE[key] = rec; _IMG_CACHE_BYTES += sz
    return rec

def predict_well_streams(rec_arrays, models, k_compute, do_tta, offsets, geom=None,
                         cache_key=None):
    """Phase-averaged per-MD-row predictions for ONE well and ONE model batch (all models from
    ONE member, so they share k_compute, instance temperatures AND image geometry).
    geom: dict(H, W, BIN_SIZE, K_MAX) — the member's render/decode geometry (None -> defaults).
    cache_key: stable well id enabling the cross-member phase-image cache.
    SPEED PATH: all phase images are stacked and forwarded as ONE batch per model (chunked at
    PHASE_CHUNK). Eval-mode convnets are batch-independent, so this equals the per-phase loop
    up to floating-point reduction order — and batched forwards are the configuration the
    §16/§17 CV probes actually validated. Returns ({stream: pred_full}, {stream: n_models}).
    Predictions are per-MD-row TVT — geometry-free — so mixed-geometry members blend directly."""
    if geom is None:
        geom = dict(H=DEFAULT_H, W=DEFAULT_W, BIN_SIZE=DEFAULT_BIN_SIZE, K_MAX=DEFAULT_K_MAX)
    _H, _W, _BS, _KM = int(geom['H']), int(geom['W']), float(geom['BIN_SIZE']), int(geom['K_MAX'])
    geom_key = (_H, _W, round(_BS, 12), _KM)
    n_hw = len(rec_arrays['hw_md'])
    bin_w = n_hw / float(_W)
    use_offsets = offsets if (do_tta and len(offsets) > 0) else [0.0]
    stream_names = [f'k{j}' for j in range(1, k_compute + 1)]
    members = {st: len(models) for st in stream_names}
    acc = {st: np.zeros(n_hw, dtype=np.float64) for st in stream_names}
    cnt = np.zeros(n_hw, dtype=np.float64)
    _stream_ckpts = str(CKPT_DEVICE) != str(DEVICE)

    recs = []
    for off in use_offsets:
        rec = _get_phase_rec(rec_arrays, cache_key, geom_key, off, _H, _W, _BS, _KM, n_hw, bin_w)
        if rec is not None: recs.append(rec)
    if not recs:
        return {}, members
    P = len(recs)
    imgs_cpu = torch.from_numpy(np.stack([r['image_in'] for r in recs]))   # (P, C, H, W)

    # per-device worker: runs ITS models serially; devices run CONCURRENTLY (thread each —
    # CUDA ops release the GIL, so two T4s overlap instead of taking turns). Cross-model
    # summation order changes vs the serial loop: fp reduction tolerance only, same math.
    def _run_models(model_group):
        local = {j: np.zeros((P, _W), dtype=np.float64) for j in range(1, k_compute + 1)}
        with torch.no_grad():
            for m in model_group:
                dev  = next(m.parameters()).device
                mm   = m.to(DEVICE) if _stream_ckpts else m
                cdev = DEVICE if _stream_ckpts else dev
                ri   = _row_idx_for(cdev, _H)
                for lo in range(0, P, PHASE_CHUNK):
                    x = imgs_cpu[lo:lo + PHASE_CHUNK].to(cdev, non_blocking=True)
                    kl = _iter_logits(mm, x, k_compute)
                    for j in range(1, k_compute + 1):
                        er = (F.softmax(kl[j], dim=1)
                              * ri[None, :, None]).sum(dim=1).cpu().numpy()
                        local[j][lo:lo + er.shape[0]] += er
                if _stream_ckpts: m.to(CKPT_DEVICE)
        return local

    er_sum = {j: np.zeros((P, _W), dtype=np.float64) for j in range(1, k_compute + 1)}
    _by_dev = {}
    for m in models:
        _by_dev.setdefault(str(next(m.parameters()).device), []).append(m)
    if len(_by_dev) > 1 and not _stream_ckpts:
        from concurrent.futures import ThreadPoolExecutor
        with ThreadPoolExecutor(max_workers=len(_by_dev)) as _ex:
            for local in _ex.map(_run_models, _by_dev.values()):
                for j in er_sum: er_sum[j] += local[j]
    else:
        for local in map(_run_models, _by_dev.values()):
            for j in er_sum: er_sum[j] += local[j]

    ones_w = np.ones(_W)
    for pi, rec in enumerate(recs):
        be = rec['bin_edges']
        widths = np.diff(be)
        for st in stream_names:
            j = int(st[1:])
            pred_col = rec['tvt_lo'] + (er_sum[j][pi] / len(models) + 0.5) * _BS
            acc[st][be[0]:be[-1]] += np.repeat(pred_col, widths)
        cnt[be[0]:be[-1]] += np.repeat(ones_w, widths)

    preds = {}
    covered = cnt > 0
    for st in stream_names:
        if members[st] == 0: continue
        p = np.full(n_hw, np.nan, dtype=np.float64)
        p[covered] = acc[st][covered] / cnt[covered]
        if not covered.all():                                # fill uncovered toe rows forward
            last = np.nan
            for r in range(n_hw):
                if covered[r]: last = p[r]
                elif not np.isnan(last): p[r] = last
            if np.isnan(p[0]):
                p[:np.argmax(covered)] = p[covered][0] if covered.any() else 0.0
        preds[st] = p
    return preds, members

print(f'PhaseJitter TTA: {"ON" if DO_PHASE_JITTER_TTA else "OFF (single-pass)"}'
      + (f' — offsets {PHASE_TTA_OFFSETS}' if DO_PHASE_JITTER_TTA else ''))
for _n, _e in RESOLVED.items():
    print(f"  [{_n}] streams k1..k{_e['eval_k']} (blending k{_e['eval_k']} at weight {_e['weight']})")


In [ ]:
# Walk test wells: per-member BATCHED ensembling of each member's k{eval_k} stream, then the
# weighted output blend. Pure model, no spatial blend.
test_hw_files = sorted(glob.glob(os.path.join(TEST_DIR, '*__horizontal_well.csv')))
print(f'Found {len(test_hw_files)} test wells')

# === dZ sanity (unchanged from the pre-fork kernel) ===
_check_hw = pd.read_csv(test_hw_files[0])
assert 'Z' in _check_hw.columns, f"Test HW CSV has no 'Z' column. Columns: {list(_check_hw.columns)}"
_post = _check_hw['TVT_input'].isna().values
_zpost = _check_hw['Z'].values[_post]
_n_finite = int(np.isfinite(_zpost).sum())
_z_std = float(np.nanstd(_zpost)) if _n_finite > 1 else 0.0
print(f"[Z sanity] first test well: {len(_zpost)} post-PS rows, {_n_finite} finite Z, std={_z_std:.3f}")
if _n_finite == 0:
    raise RuntimeError('Post-PS Z entirely missing in test — dZ channel degenerate; do not submit this model.')
if _z_std < 1e-6:
    print('  WARNING: post-PS Z ~constant — dZ channel ~flat.')

# --- Pass 1: cache raw arrays + geometry per well (shared by every member) --------------------
well_recs = {}; well_order = []; post_idx = {}; last_known_tvt = {}
for hw_path in test_hw_files:
    well_id = os.path.basename(hw_path).split('__')[0]
    tw_path = hw_path.replace('__horizontal_well.csv', '__typewell.csv')
    if not os.path.exists(tw_path):
        print(f'  {well_id}: MISSING typewell CSV, skipping'); continue
    hw = pd.read_csv(hw_path); tw = pd.read_csv(tw_path)
    well_recs[well_id] = dict(
        hw_md=hw['MD'].values, hw_gr=hw['GR'].values, hw_z=hw['Z'].values,
        hw_tvt=np.full(len(hw), np.nan, dtype=np.float64), hw_tvti=hw['TVT_input'].values,
        tw_tvt=tw['TVT'].values, tw_gr=tw['GR'].values)
    well_order.append(well_id)
    post_idx[well_id] = np.where(hw['TVT_input'].isna().values)[0]
    known = hw['TVT_input'].dropna()
    last_known_tvt[well_id] = float(known.iloc[-1]) if len(known) > 0 else float('nan')
print(f'Cached {len(well_order)} wells; running {len(RESOLVED)} ensemble members.')

# --- Pass 2: per-member batched accumulation --------------------------------------------------
# MEMBER_STREAMS[name][well][stream] = that member's ckpt-mean, TTA-averaged prediction (n_hw,).
# Only the member's k{eval_k} stream enters the blend; lower iterates are free diagnostics.
MEMBER_STREAMS = {}
for name, entry in RESOLVED.items():
    ckpt_paths = entry['ckpt_paths']
    head_st = f"k{entry['eval_k']}"          # this member's headline (blended) stream
    _mbs = member_batch_size(entry)
    n_batches = (len(ckpt_paths) + _mbs - 1) // _mbs
    _g = entry['geom']
    print(f"\n[{name}] {len(ckpt_paths)} ckpts, eval_k={entry['eval_k']}, "
          f"handoff_t={entry['handoff_t']}, refeed_t={entry['refeed_t']}, "
          f"geom={_g['H']}x{_g['W']}@bin{_g['BIN_SIZE']:.4f} -> {n_batches} batches "
          f"(auto batch {_mbs}, ckpt ~{entry.get('_ckpt_gib', float('nan')):.2f} GiB, "
          f"{len(GPUS)} GPU{'s' if len(GPUS) != 1 else ''})")
    stream_sum = {w: {} for w in well_order}     # well -> stream -> weighted pred sum
    stream_n   = {}                              # stream -> total member models accumulated
    for b in range(n_batches):
        lo = b * _mbs
        paths = ckpt_paths[lo:lo + _mbs]
        batch_models = _load_model_batch(entry, paths)
        n_loaded = len(batch_models)
        members_seen = None
        for well_id in well_order:
            preds, members = predict_well_streams(well_recs[well_id], batch_models,
                                                  entry['eval_k'],
                                                  DO_PHASE_JITTER_TTA, PHASE_TTA_OFFSETS,
                                                  geom=entry['geom'], cache_key=well_id)
            members_seen = members
            for st, p in preds.items():
                prev = stream_sum[well_id].get(st)
                stream_sum[well_id][st] = p * members[st] if prev is None else prev + p * members[st]
        for st, mcount in members_seen.items():
            stream_n[st] = stream_n.get(st, 0) + mcount
        _free_model_batch(batch_models)
        print(f"  batch {b + 1}/{n_batches}: +{n_loaded} ckpts "
              f"({stream_n[head_st]}/{len(ckpt_paths)} on {head_st}), freed")
    assert stream_n[head_st] == len(ckpt_paths), \
        f'[{name}] accumulated ckpt count != resolved ckpt count'
    MEMBER_STREAMS[name] = {w: {st: stream_sum[w][st] / float(stream_n[st])
                                for st in stream_sum[w]} for w in well_order}
    # cross-member image-cache bookkeeping: purge this geometry once its LAST member is done
    _gk = (entry['geom']['H'], entry['geom']['W'], round(entry['geom']['BIN_SIZE'], 12),
           entry['geom']['K_MAX'])
    _later = [RESOLVED[_n2] for _n2 in list(RESOLVED)[list(RESOLVED).index(name) + 1:]]
    if not any((e2['geom']['H'], e2['geom']['W'], round(e2['geom']['BIN_SIZE'], 12),
                e2['geom']['K_MAX']) == _gk for e2 in _later):
        _drop = [k for k in list(_IMG_CACHE) if k[1] == _gk]
        for k in _drop: del _IMG_CACHE[k]
        import sys as _sys
        globals()['_IMG_CACHE_BYTES'] = sum(
            v['image_in'].nbytes + v['bin_edges'].nbytes + 64 for v in _IMG_CACHE.values())
        if _drop: print(f"  image cache: purged {len(_drop)} renders for geometry {_gk[:2]}")
    print(f"  image cache: {_IMG_STATS['built']} built / {_IMG_STATS['hits']} reused so far "
          f"({_IMG_CACHE_BYTES/2**30:.2f} GiB held)")

# --- Blend: weighted convex combination of each member's k{eval_k} stream ---------------------
# GUARD (defense in depth): the blend is only valid if weights sum to EXACTLY 1.0.
_wsum = float(sum(e['weight'] for e in RESOLVED.values()))
if _wsum != 1.0:
    raise ValueError(f'Blend weights must sum to EXACTLY 1.0 at blend time — got {_wsum!r}')

BLEND_PRED = {}
for well_id in well_order:
    n_hw = len(well_recs[well_id]['hw_md'])
    acc = np.zeros(n_hw, dtype=np.float64)
    for name, entry in RESOLVED.items():
        acc += entry['weight'] * MEMBER_STREAMS[name][well_id][f"k{entry['eval_k']}"]
    BLEND_PRED[well_id] = acc

predictions = {}
_nan_rows = 0
for well_id in well_order:
    pred_final = BLEND_PRED[well_id]
    for ri in post_idx[well_id]:
        v = float(pred_final[ri])
        if not np.isfinite(v):
            _nan_rows += 1
        predictions[(well_id, int(ri))] = v
if _nan_rows > 0:
    print(f'  WARNING: {_nan_rows} blended rows are NaN (a member left them uncovered) — the '
          f'submission fallback will fill them; inspect if this is more than a handful.')

_desc = ' + '.join(f"{e['weight']}*{n}[k{e['eval_k']},hT{e['handoff_t']}]"
                   for n, e in RESOLVED.items())
print(f'\nTotal predictions: {len(predictions):,}  (blend = {_desc}'
      + (', TTA' if DO_PHASE_JITTER_TTA else '') + ', pure model)')


In [ ]:
# Build submission in the order specified by sample_submission.csv (predictions = the blend)
sample_sub = pd.read_csv(SAMPLE_SUB)
print(f'Sample submission has {len(sample_sub):,} rows, columns: {sample_sub.columns.tolist()}')

# Drop the placeholder tvt column from the sample so it doesn't shadow our predictions in the merge.
# (Earlier bug: with suffixes=('', '_pred'), the left tvt column kept its name and our predictions
#  ended up in tvt_pred. Sample-submission zeros were then written to submission.csv.)
sample_sub_ordered = sample_sub[['id']].copy()

# Parse well_id and md_row from id column (id format: '{well_id}_{md_row_0_based}')
parts = sample_sub_ordered['id'].str.rsplit('_', n=1, expand=True)
sample_sub_ordered['well_id'] = parts[0]
sample_sub_ordered['md_row']  = parts[1].astype(int)

# Convert the predictions dict to a DataFrame for a fast merge
pred_records = [{'well_id': w, 'md_row': r, 'tvt': v} for (w, r), v in predictions.items()]
pred_df = pd.DataFrame(pred_records)
print(f'Predictions DataFrame: {len(pred_df):,} rows')

# Merge predictions into submission order (no column conflict now, tvt comes from pred_df)
merged = sample_sub_ordered.merge(pred_df, on=['well_id', 'md_row'], how='left')

# Apply fallback for any unmatched rows (shouldn't happen if test wells match the submission template)
n_missing = int(merged['tvt'].isna().sum())
if n_missing > 0:
    print(f'  {n_missing} rows had no model prediction; filling with per-well last-known TVT')
    fallback_map = pd.Series(last_known_tvt)
    merged.loc[merged['tvt'].isna(), 'tvt'] = merged.loc[merged['tvt'].isna(), 'well_id'].map(fallback_map)

n_still_nan = int(merged['tvt'].isna().sum())
if n_still_nan > 0:
    print(f'  {n_still_nan} rows still NaN after fallback; filling with global mean')
    merged['tvt'] = merged['tvt'].fillna(merged['tvt'].mean())

# Sanity checks
n_total = len(merged)
n_from_model = n_total - n_missing
print(f'\nSubmission rows: {n_total:,}')
print(f'  From model predictions: {n_from_model:,} ({100*n_from_model/n_total:.2f}%)')
print(f'  From fallback: {n_missing:,} ({100*n_missing/n_total:.2f}%)')

# Hard sanity check: predictions shouldn't be all zero (catches the bug we just fixed)
if merged['tvt'].abs().max() < 1.0:
    raise RuntimeError(
        f'Sanity check FAILED: max |tvt| < 1.0 across all rows. '
        f'Predictions are essentially zero. Inspect the merge.'
    )

# Write submission with the original columns only
out = merged[['id', 'tvt']]
out.to_csv('submission.csv', index=False)
print(f'\nWrote submission.csv with {len(out):,} rows')
print(f'\nFirst rows:')
print(out.head().to_string())
print(f'\nLast rows:')
print(out.tail().to_string())
print(f'\nPrediction stats: min={out["tvt"].min():.2f}, mean={out["tvt"].mean():.2f}, max={out["tvt"].max():.2f}')



In [ ]:
# Quick visual sanity check: per-well prediction trajectories — blend + each member's headline
import matplotlib.pyplot as plt

n_wells_to_plot = min(3, len(test_hw_files))
fig, axes = plt.subplots(1, n_wells_to_plot, figsize=(5 * n_wells_to_plot, 4))
if n_wells_to_plot == 1:
    axes = [axes]

for ax, hw_path in zip(axes, test_hw_files[:n_wells_to_plot]):
    well_id = os.path.basename(hw_path).split('__')[0]
    hw = pd.read_csv(hw_path, usecols=['MD', 'TVT_input'])
    is_post_ps = hw['TVT_input'].isna().values
    md_indices_post = np.where(is_post_ps)[0]
    md_indices_pre  = np.where(~is_post_ps)[0]
    preds = np.array([predictions.get((well_id, int(ri)), np.nan) for ri in md_indices_post])

    ax.plot(hw.loc[md_indices_pre, 'MD'].values, hw.loc[md_indices_pre, 'TVT_input'].values,
            'b-', lw=1.5, label='TVT_input (known)')
    if well_id in well_order:
        for name, entry in RESOLVED.items():
            mp = MEMBER_STREAMS[name][well_id][f"k{entry['eval_k']}"][md_indices_post]
            ax.plot(hw.loc[md_indices_post, 'MD'].values, mp, ls='--', lw=0.9, alpha=0.7,
                    label=f"{name} (w={entry['weight']})")
    ax.plot(hw.loc[md_indices_post, 'MD'].values, preds,
            'r-', lw=1.8, label='ENSEMBLE blend (submitted)')

    last_known = last_known_tvt.get(well_id, np.nan)
    if not np.isnan(last_known):
        ax.axhline(last_known, color='gray', ls=':', alpha=0.5, label='last known value')
    ax.set_title(f'{well_id}: {len(md_indices_post)} predicted rows')
    ax.set_xlabel('MD'); ax.set_ylabel('TVT')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

# Sanity: predicted values should be in a sensible range relative to last-known
print('\nPer-well prediction sanity (blend):')
for hw_path in test_hw_files[:5]:
    well_id = os.path.basename(hw_path).split('__')[0]
    last_known = last_known_tvt.get(well_id, np.nan)
    preds_well = [v for (w, _), v in predictions.items() if w == well_id]
    if preds_well:
        arr = np.array(preds_well)
        print(f'  {well_id}: last known TVT = {last_known:.2f}, '
              f'pred range = [{arr.min():.2f}, {arr.max():.2f}], '
              f'pred mean = {arr.mean():.2f}, deviation from last = {arr.mean() - last_known:+.2f}')

# Member-vs-blend divergence audit: how far apart are the members where it matters (post-PS)?
print('\nMember divergence audit (post-PS rows, mean |member - blend| in ft):')
for name, entry in RESOLVED.items():
    diffs = []
    for well_id in well_order:
        mp = MEMBER_STREAMS[name][well_id][f"k{entry['eval_k']}"]
        bp = BLEND_PRED[well_id]
        idx = post_idx[well_id]
        d = np.abs(mp[idx] - bp[idx])
        diffs.append(d[np.isfinite(d)])
    d_all = np.concatenate(diffs) if diffs else np.array([0.0])
    print(f"  [{name}] w={entry['weight']}: mean {d_all.mean():.3f}, "
          f"p95 {np.percentile(d_all, 95):.3f}, max {d_all.max():.3f}")
